https://github.com/tylerebowers/Schwabdev

In [20]:
pip install cryptography

   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ------------------ --------------------- 1.6/3.5 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 12.9 MB/s  0:00:00

   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   ---------------------------------------- 3/3 [cryptography]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trio 0.29.0 requires sniffio>=1.3.0, which is not installed.
trio 0.29.0 requires sortedcontainers, which is not installed.


In [ ]:
from schwabdev import client
import os
import logging
import pandas as pd
from dotenv import load_dotenv

C:\Users\guloo\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(


In [ ]:
# set logging level
logging.basicConfig(level=logging.INFO)

# load environment variables from parent folder and make client
load_dotenv('../.env')
client = client.Client(os.getenv('app_key'), os.getenv('app_secret'))

[Schwabdev] Open to authenticate: https://api.schwabapi.com/v1/oauth/authorize?client_id=cICd6Ljp6kgcQyxIybtFyifJRxK4CbLych6TotAAymodTC0k&redirect_uri=https://127.0.0.1


In [ ]:
ticker_list = ['/GC', 'GLD', '/SI', 'SLV', '/BTC', '/ETH', '/CL', '/NG']
frequency_types = ['daily', 'weekly', 'monthly']
base_out_dir = './data'

In [ ]:
def get_price_history(symbol, frequency_type, periodType='year', period=20, frequency=1):
    response = client.price_history(
        symbol=symbol,
        periodType=periodType,
        period=period,
        frequencyType=frequency_type,
        frequency=frequency,
        needExtendedHoursData=False,
        needPreviousClose=True
    )
    df = pd.json_normalize(response.json()['candles'])
    df['datetime'] = pd.to_datetime(df['datetime'], unit='ms', utc=True)
    return df


def update_price_data(ticker, frequency_type, base_out_dir='./data', periodType='year', period=20, frequency=1):
    out_dir = os.path.join(base_out_dir, frequency_type)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'{ticker.replace("/", "_")}.parquet')

    existing = None
    if os.path.exists(out_path):
        existing = pd.read_parquet(out_path)
        existing['datetime'] = pd.to_datetime(existing['datetime'], utc=True)
        existing = existing.sort_values('datetime')
        last_datetime = existing['datetime'].max()
    else:
        last_datetime = None

    df = get_price_history(ticker, frequency_type, periodType=periodType, period=period, frequency=frequency)
    df = df.sort_values('datetime')

    if last_datetime is not None:
        df_new = df[df['datetime'] > last_datetime]
    else:
        df_new = df

    if existing is not None:
        combined = pd.concat([existing, df_new], ignore_index=True)
        combined = combined.drop_duplicates(subset=['datetime']).sort_values('datetime')
    else:
        combined = df

    if existing is None or len(combined) > len(existing):
        combined.to_parquet(out_path, engine='pyarrow', index=False)
        added = len(combined) - (len(existing) if existing is not None else 0)
        print(f'Updated {out_path}: added {added} new row(s), total {len(combined)} rows')
    else:
        print(f'No new data for {out_path}. Total rows remain {len(combined)}')

    return combined


for frequency_type in frequency_types:
    for ticker in ticker_list:
        update_price_data(ticker, frequency_type, base_out_dir)
